In [1]:
import xarray as xr
import pandas as pd

ds1 = xr.open_dataset("rf_ctrl_hourly_7-10july2023.nc") 
print("CTRL:")
print(ds1) 
print("_"*100)                 

CTRL:
<xarray.Dataset> Size: 218MB
Dimensions:                        (time: 73, lat: 635, lon: 589)
Coordinates:
  * time                           (time) datetime64[ns] 584B 2023-07-07 ... ...
  * lat                            (lat) float64 5kB 26.01 26.02 ... 36.97 36.99
  * lon                            (lon) float64 5kB 71.02 71.04 ... 82.97 82.99
Data variables:
    __xarray_dataarray_variable__  (time, lat, lon) float64 218MB ...
____________________________________________________________________________________________________


In [2]:
var = ds1["__xarray_dataarray_variable__"]
print(var['time'])

<xarray.DataArray 'time' (time: 73)> Size: 584B
array(['2023-07-07T00:00:00.000000000', '2023-07-07T01:00:00.000000000',
       '2023-07-07T02:00:00.000000000', '2023-07-07T03:00:00.000000000',
       '2023-07-07T04:00:00.000000000', '2023-07-07T05:00:00.000000000',
       '2023-07-07T06:00:00.000000000', '2023-07-07T07:00:00.000000000',
       '2023-07-07T08:00:00.000000000', '2023-07-07T09:00:00.000000000',
       '2023-07-07T10:00:00.000000000', '2023-07-07T11:00:00.000000000',
       '2023-07-07T12:00:00.000000000', '2023-07-07T13:00:00.000000000',
       '2023-07-07T14:00:00.000000000', '2023-07-07T15:00:00.000000000',
       '2023-07-07T16:00:00.000000000', '2023-07-07T17:00:00.000000000',
       '2023-07-07T18:00:00.000000000', '2023-07-07T19:00:00.000000000',
       '2023-07-07T20:00:00.000000000', '2023-07-07T21:00:00.000000000',
       '2023-07-07T22:00:00.000000000', '2023-07-07T23:00:00.000000000',
       '2023-07-08T00:00:00.000000000', '2023-07-08T01:00:00.000000000',
   

In [3]:
train_split = var.isel(time=slice(0, 60)).rename("__xarray_dataarray_variable__")
encoding = {"__xarray_dataarray_variable__": {"zlib": True, "complevel": 4}}
train_split.to_netcdf("custom_data/train/data.nc", encoding=encoding)

test_split = var.isel(time=slice(60, None)).rename("__xarray_dataarray_variable__")
encoding = {"__xarray_dataarray_variable__": {"zlib": True, "complevel": 4}}
test_split.to_netcdf("custom_data/test/data.nc", encoding=encoding)

In [4]:
ds1 = xr.open_dataset("custom_data/test/data.nc") 
print(ds1)

<xarray.Dataset> Size: 39MB
Dimensions:                        (time: 13, lat: 635, lon: 589)
Coordinates:
  * time                           (time) datetime64[ns] 104B 2023-07-09T12:0...
  * lat                            (lat) float64 5kB 26.01 26.02 ... 36.97 36.99
  * lon                            (lon) float64 5kB 71.02 71.04 ... 82.97 82.99
Data variables:
    __xarray_dataarray_variable__  (time, lat, lon) float64 39MB ...


In [5]:
# Build a 256-entry PIL palette matching SEVIR VIL bins
import sys
import numpy as np
from PIL import Image
from IPython.display import display

# Ensure local package is importable
if 'src' not in sys.path:
    sys.path.append('src')

from prediff.datasets.sevir.sevir_cmap import VIL_COLORS, VIL_LEVELS

# Recompute 0–255 scaled data (independent of previous cells)
for i in range(0, var.shape[0]):
    base = var[i].astype(np.float32)
    arr = base.values if hasattr(base, "values") else np.asarray(base, dtype=np.float32)
    amin = np.nanmin(arr)
    amax = np.nanmax(arr)
    ptp = amax - amin
    scaled = (arr - amin) / (ptp if ptp > 0 else 1.0)
    arr_u8 = np.uint8(np.clip(scaled * 255, 0, 255))

    # Skip the first color (nil/bad) and keep the 10 bin colors
    rgb_cols = [(int(round(r * 255)), int(round(g * 255)), int(round(b * 255)))
                for (r, g, b) in VIL_COLORS[1:11]]
    levels = VIL_LEVELS

    palette = []
    for v in range(256):
        # Find which bin this value falls into
        idx = next((i for i in range(len(levels) - 1) if levels[i] <= v < levels[i + 1]), len(levels) - 2)
        palette.extend(rgb_cols[idx])

    # Ensure palette length is exactly 768 entries
    if len(palette) < 768:
        palette.extend([0, 0, 0] * (256 - len(palette) // 3))

    im_pal = Image.fromarray(arr_u8, mode="P")
    im_pal.putpalette(palette)
    # display(im_pal)
    # Optional save
    im_pal.save(f"processed/img_{i}.png")

C:\Users\dhruv\AppData\Local\Temp\ipykernel_84796\1342974240.py:38: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  im_pal = Image.fromarray(arr_u8, mode="P")
